In [0]:
from pyspark.sql import functions as F

database_name = "fsdh_artemis"

raw_file_path = "/mnt/fsdh-dbk-main-mount/Orion_Only_CURRENT_Trajectory.asc"

bronze_table_file = f"{database_name}.bronze_oem_file_registry"
bronze_table_lines = f"{database_name}.bronze_oem_lines"

In [0]:
display(dbutils.fs.ls("/mnt/fsdh-dbk-main-mount"))

In [0]:
raw_file_df = (
    spark.read.format("binaryFile")
    .load(raw_file_path)
    .select(
        F.col("path").alias("source_file"),
        F.col("modificationTime").alias("file_modified_ts"),
        F.col("length").alias("file_size_bytes"),
        F.sha2(F.col("content"), 256).alias("file_sha256"),
        F.decode(F.col("content"), "UTF-8").alias("file_text")
    )
)

display(raw_file_df)

In [0]:
raw_file_df.write.mode("overwrite").saveAsTable(bronze_table_file)

print(f"Saved table: {bronze_table_file}")
display(spark.table(bronze_table_file))

In [0]:
raw_lines_df = (
    raw_file_df
    .withColumn("line_array", F.split(F.col("file_text"), r"\r?\n"))
    .select(
        "source_file",
        "file_modified_ts",
        "file_size_bytes",
        "file_sha256",
        F.posexplode("line_array").alias("line_number", "raw_line")
    )
    .withColumn("line_number", F.col("line_number") + 1)
    .withColumn("ingested_at", F.current_timestamp())
)

display(raw_lines_df)

In [0]:
raw_lines_df.write.mode("overwrite").saveAsTable(bronze_table_lines)

print(f"Saved table: {bronze_table_lines}")
display(
    spark.table(bronze_table_lines)
    .orderBy("line_number")
)